# Custom Dataset + Paradigm A: Supervised Task Optimization

This notebook demonstrates how to import **user-generated data** into the NeuralRNN framework
and train a model using Paradigm A (supervised task optimization).

**Scenario**: You have experimental or simulated data where:
- **Input**: 2-dimensional signal, 2000 timepoints, 100 trials → array shape `(2000, 2, 100)`
- **Output**: 1-dimensional signal, 2000 timepoints, 100 trials → array shape `(2000, 1, 100)`

We will:
1. Generate synthetic data matching your format
2. Import it via `CustomDataset`
3. Configure and train a CTRNN model
4. Evaluate the trained model

## 0. Setup

In [2]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import sys
sys.path.append('../src')

from neuralrnn import (
    AutoConfig, AutoModel,
    Trainer, TrainingArguments,
    SupervisedObjective,
)
from neuralrnn.data.custom_dataset import CustomDataset

## 1. Prepare Your Data

Your data convention: `(timepoints, features, trials)`.

For NeuralRNN, we need: `(trials, timepoints, features)` — batch-first.

Below we generate synthetic data that mimics your format. Replace this with your actual data loading.

In [3]:
# ======================================================================
# Replace this block with your actual data loading.
# Expected shapes:
#   inputs_raw:  (T, input_dim, n_trials) = (2000, 2, 100)
#   outputs_raw: (T, output_dim, n_trials) = (2000, 1, 100)
# ======================================================================

np.random.seed(42)

T = 2000        # timepoints
input_dim = 2   # 2D input
output_dim = 1  # 1D output
n_trials = 100  # number of trials

# Synthetic: output is a filtered version of input + noise
inputs_raw = np.random.randn(T, input_dim, n_trials).astype(np.float32) * 0.5

# Simple linear mapping + smoothing + noise (so the model has something to learn)
from scipy.ndimage import uniform_filter1d
signal = uniform_filter1d(inputs_raw[:, 0, :], size=20, axis=0)  # smooth first dim
outputs_raw = (signal[:, np.newaxis, :] + 0.1 * np.random.randn(T, output_dim, n_trials)).astype(np.float32)

print(f"inputs_raw shape:  {inputs_raw.shape}  (T={T}, input_dim={input_dim}, n_trials={n_trials})")
print(f"outputs_raw shape: {outputs_raw.shape}  (T={T}, output_dim={output_dim}, n_trials={n_trials})")

inputs_raw shape:  (2000, 2, 100)  (T=2000, input_dim=2, n_trials=100)
outputs_raw shape: (2000, 1, 100)  (T=2000, output_dim=1, n_trials=100)


## 2. Import into CustomDataset

Key step: **transpose** from `(T, D, trials)` → `(trials, T, D)` to match NeuralRNN's batch-first convention.

In [13]:
# Transpose to (trials, timepoints, features) — NeuralRNN batch-first convention
inputs = inputs_raw.transpose(2, 0, 1)    # (100, 2000, 2)
outputs = outputs_raw.transpose(2, 0, 1)  # (100, 2000, 1)

print(f"After transpose:")
print(f"  inputs:  {inputs.shape}  (n_trials, T, input_dim)")
print(f"  outputs: {outputs.shape}  (n_trials, T, output_dim)")

# Create CustomDataset
dataset = CustomDataset.from_arrays(
    inputs=inputs,
    targets=outputs,
    mode="supervised",      # Paradigm A: input-output pairs
    batch_size=16,
    normalize=False,         # set True if input scales vary widely
    test_fraction=0.2,       # hold out 20% of trials for evaluation
    seed=42,
)

print(f"\nDataset created:")
print(f"  mode: {dataset.mode}")
print(f"  input_dim:  {dataset.input_dim}")
print(f"  output_dim: {dataset.output_dim}")
print(f"  n_train_trials: {len(dataset)}")
print(f"  test_set available: {dataset.test_set is not None}")
if dataset.test_set is not None:
    print(f"  n_test_trials: {len(dataset.test_set)}")

After transpose:
  inputs:  (100, 2000, 2)  (n_trials, T, input_dim)
  outputs: (100, 2000, 1)  (n_trials, T, output_dim)

Dataset created:
  mode: supervised
  input_dim:  2
  output_dim: 1
  n_train_trials: 80
  test_set available: True
  n_test_trials: 20


In [14]:
# Inspect a sample batch
batch = dataset.sample_batch()
print("Batch keys:", list(batch.keys()))
for k, v in batch.items():
    if v is not None:
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
    else:
        print(f"  {k}: None")

Batch keys: ['inputs', 'targets', 'mask']
  inputs: shape=torch.Size([16, 2000, 2]), dtype=torch.float32
  targets: shape=torch.Size([16, 2000, 1]), dtype=torch.float32
  mask: shape=torch.Size([16, 2000]), dtype=torch.float32


## 3. Configure and Build the Model

We use a CTRNN (continuous-time RNN) for this supervised task. Key config:
- `input_dim=2`: matches your 2D input
- `output_dim=1`: matches your 1D output (regression readout)
- `latent_dim=64`: hidden units (tune as needed)

In [15]:
config = AutoConfig.for_model(
    "ctrnn",
    input_dim=dataset.input_dim,    # 2
    latent_dim=4,                   # hidden units
    output_dim=dataset.output_dim,  # 1 (regression)
    dt=1.0,                         # Euler step
    tau=10.0,                       # time constant
    activation="relu",
    trainable_h0=False,
)
model = AutoModel.from_config(config)
print(f"Model: {type(model).__name__}")
print(f"Parameters: {model.num_parameters():,}")
print(config)

Model: CTRNNModel
Parameters: 37
CTRNNConfig({
  "activation": "relu",
  "dale": false,
  "dt": 1.0,
  "ei_ratio": 0.8,
  "input_dim": 2,
  "latent_dim": 4,
  "model_type": "ctrnn",
  "output_dim": 1,
  "sigma_rec": 0.0,
  "tau": 10.0,
  "trainable_h0": false
})


## 4. Train

Use `SupervisedObjective(task_type="regression")` with MSE loss.
The Trainer calls `dataset.sample_batch()` each step, feeds inputs to the model,
and backprops the loss between readout outputs and targets.

In [16]:
objective = SupervisedObjective(task_type="regression")

args = TrainingArguments(
    learning_rate=1e-3,
    max_steps=100,
    batch_size=16,
    grad_clip_norm=1.0,
    optimizer="adam",
    log_every=20,
    device="cuda" if torch.cuda.is_available() else "cpu",
    seed=42,
)

trainer = Trainer(
    model=model,
    dataset=dataset,
    objective=objective,
    args=args,
)

print(f"Training on: {args.device}")
print(f"Max steps: {args.max_steps}")
print(f"Objective: {type(objective).__name__} ({objective.task_type})")

Training on: cuda
Max steps: 100
Objective: SupervisedObjective (regression)


In [17]:
history = trainer.train()

[train] step=0  loss=0.2456
[train] step=20  loss=0.1528
[train] step=40  loss=0.0861
[train] step=60  loss=0.0430
[train] step=80  loss=0.0247


In [ ]:
# Plot training loss
steps = [h["step"] for h in history]
losses = [h["loss"] for h in history]

plt.figure(figsize=(8, 4))
plt.plot(steps, losses, linewidth=1.5)
plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.title("Training Loss")
plt.yscale("log")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Evaluate on Test Set

Run the trained model on held-out test trials and compare predictions vs ground truth.

In [19]:
model.eval()

# Get test data (full trials)
test_ds = dataset.test_set
test_inputs = test_ds.X   # (n_test, T, 2)
test_targets = test_ds.Y  # (n_test, T, 1)

print(f"Test inputs:  {test_inputs.shape}")
print(f"Test targets: {test_targets.shape}")

# Run model forward on all test trials at once
with torch.no_grad():
    device = next(model.parameters()).device
    out = model(test_inputs.to(device))
    preds = out.outputs.cpu()  # (n_test, T, 1)

print(f"Predictions: {preds.shape}")

Test inputs:  torch.Size([20, 2000, 2])
Test targets: torch.Size([20, 2000, 1])
Predictions: torch.Size([20, 2000, 1])


In [ ]:
# Plot a few test trials: prediction vs ground truth
n_show = min(4, test_inputs.shape[0])
fig, axes = plt.subplots(n_show, 1, figsize=(12, 3 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    t = np.arange(test_targets.shape[1])
    ax.plot(t, test_targets[i, :, 0].numpy(), label="Ground truth", alpha=0.8)
    ax.plot(t, preds[i, :, 0].numpy(), label="Prediction", alpha=0.8)
    ax.set_ylabel("Output")
    ax.legend(loc="upper right")
    ax.set_title(f"Test trial {i}")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Timepoint")
plt.suptitle("Model Predictions vs Ground Truth (Test Set)", y=1.01)
plt.tight_layout()
plt.show()

In [21]:
# Compute test MSE
mse = ((preds - test_targets) ** 2).mean().item()
print(f"Test MSE: {mse:.6f}")

# Per-trial MSE
per_trial_mse = ((preds - test_targets) ** 2).mean(dim=(1, 2))  # (n_test,)
print(f"Per-trial MSE: mean={per_trial_mse.mean():.6f}, std={per_trial_mse.std():.6f}")

Test MSE: 0.019955
Per-trial MSE: mean=0.019955, std=0.001326


## 6. Save and Reload the Model

In [ ]:
# Save
save_dir = "runs/ctrnn_custom_dataset"
model.save_pretrained(save_dir)
print(f"Model saved to {save_dir}/")

# Reload
model_reloaded = AutoModel.from_pretrained(save_dir)
print(f"Model reloaded: {type(model_reloaded).__name__}")

# Verify: outputs should match
with torch.no_grad():
    out2 = model_reloaded(test_inputs)
    preds2 = out2.outputs
diff = (preds - preds2).abs().max().item()
print(f"Max difference after reload: {diff:.2e}")

## Summary

**How to use your own data with Paradigm A:**

```python
from neuralrnn.data.custom_dataset import CustomDataset

# 1. Load your data as numpy arrays
#    Convention: (timepoints, features, trials)
inputs_raw  = ...  # (2000, 2, 100)
outputs_raw = ...  # (2000, 1, 100)

# 2. Transpose to (trials, timepoints, features)
inputs  = inputs_raw.transpose(2, 0, 1)   # (100, 2000, 2)
outputs = outputs_raw.transpose(2, 0, 1)  # (100, 2000, 1)

# 3. Create dataset
dataset = CustomDataset.from_arrays(
    inputs=inputs, targets=outputs,
    mode="supervised", batch_size=16,
)

# 4. Train
model = AutoModel.from_config(AutoConfig.for_model("ctrnn",
    input_dim=2, latent_dim=64, output_dim=1))
Trainer(model, dataset, SupervisedObjective("regression"),
        TrainingArguments(max_steps=3000)).train()
```

**Other data loading options:**
- `CustomDataset.from_npz("data.npz")` — load from `.npz` file
- `CustomDataset.from_mat("data.mat")` — load from MATLAB `.mat` file
- `CustomDataset.from_dict({"inputs": arr1, "targets": arr2})` — load from dict